<a href="https://colab.research.google.com/github/asesofspade/STT-REAL-TIME-TRANSCRIBER/blob/main/STT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai-whisper
!pip install -q scipy numpy
!apt-get install -q ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 25.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from IPython.display import display, Javascript
from google.colab import output
import base64, tempfile, os
import whisper
from pydub import AudioSegment

print("Loading whisper large...")
model = whisper.load_model("large")
print("Ready\n")

HALLUCINATIONS = {
    "gracias", "gracias.", "thank you", "thank you.", "thanks",
    "subtítulos", "suscríbete", "bye", "adiós", "...", ".", "",
    "intervoices.com", "www.", "¿qué es la educación?", "¿qué es la educación"
}

recordings = []

def pad_audio(path, pad_seconds=2):
    audio = AudioSegment.from_file(path, format="webm")
    silence = AudioSegment.silent(duration=pad_seconds * 1000)
    padded = silence + audio + silence
    wav_path = path.replace(".webm", ".wav")
    padded.export(wav_path, format="wav")
    return wav_path

def save_chunk(data):
    raw = base64.b64decode(data.split(',')[1])
    with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as f:
        f.write(raw)
        recordings.append(f.name)
    print(f"Recording {len(recordings)} saved")

def transcribe_all(_=None):
    if not recordings:
        print("NULL RECORDINGS.")
        return
    print(f"\nTranscribing {len(recordings)} recordings...\n")
    results = []
    for i, path in enumerate(recordings):
        if not os.path.exists(path):
            continue
        try:
            wav_path = pad_audio(path, pad_seconds=2)
        except Exception as e:
            print(f"  [{i+1}] ERROR PROCESSING INPUT: {e}")
            os.unlink(path)
            continue
        os.unlink(path)

        result = model.transcribe(
            wav_path,
            language="es",
            fp16=True,
            temperature=0.0,
            no_speech_threshold=0.6,
            logprob_threshold=-0.8,
            compression_ratio_threshold=2.0,
            condition_on_previous_text=False,
            initial_prompt="Spanish transcription:"
        )
        os.unlink(wav_path)

        segments = result.get("segments", [])
        if not segments:
            print(f"  [{i+1}] (VOID)")
            continue

        avg_no_speech = sum(s["no_speech_prob"] for s in segments) / len(segments)
        avg_logprob   = sum(s["avg_logprob"]    for s in segments) / len(segments)

        if avg_no_speech > 0.5 or avg_logprob < -0.8:
            print(f"  [{i+1}] (Skipped - low trustness)")
            continue

        text = result["text"].strip()
        clean = "".join(c for c in text if ord(c) < 1000 or c in "áéíóúüñÁÉÍÓÚÜÑ¿¡").strip()
        print(f"  [{i+1}] {clean if clean else '(VOID)'}")

        if clean and clean.lower() not in HALLUCINATIONS and len(clean) > 3:
            results.append(clean)

    recordings.clear()
    final = ". ".join(results)
    if final:
        print(f"\n {final}\n")
    else:
        print("NILL.\n")

output.register_callback('notebook.save_chunk', save_chunk)
output.register_callback('notebook.transcribe_all', transcribe_all)

display(Javascript('''
let liveStream = null;
let liveRecording = false;
let muted = false;
let mediaRecorder = null;
let chunks = [];
let silenceTimer = null;
let speaking = false;
let analyser = null;
let animFrame = null;
let audioCtx = null;

const SILENCE_DELAY_MS = 600;
const VOLUME_THRESHOLD = 6;
const SPEECH_RATIO     = 0.04;

const btn = document.createElement("button");
btn.innerText = "Init";
btn.style = "padding:10px 20px; font-size:16px; cursor:pointer; border-radius:8px; margin-right:8px;";
document.body.appendChild(btn);

const muteBtn = document.createElement("button");
muteBtn.innerText = "MUTE";
muteBtn.style = "padding:10px 20px; font-size:16px; cursor:pointer; border-radius:8px; margin-right:8px; display:none;";
document.body.appendChild(muteBtn);

const transcribeBtn = document.createElement("button");
transcribeBtn.innerText = "Transcribe all";
transcribeBtn.style = "padding:10px 20px; font-size:16px; cursor:pointer; border-radius:8px; background:#2196F3; color:white; border:none; display:none;";
document.body.appendChild(transcribeBtn);

const status = document.createElement("span");
status.style = "margin-left:12px; font-size:14px; color:gray;";
document.body.appendChild(status);

function setMicEnabled(enabled) {
    if (!liveStream) return;
    liveStream.getAudioTracks()[0].enabled = enabled;
}

muteBtn.onclick = () => {
    muted = !muted;
    setMicEnabled(!muted);
    muteBtn.innerText = muted ? "UNMUTE" : "MUTE";
    status.innerText = muted ? "MUTED" : "Listening...";
    if (muted && speaking) stopSpeech(false);
};

transcribeBtn.onclick = async () => {
    liveRecording = false;
    cancelAnimationFrame(animFrame);
    if (speaking) stopSpeech(false);
    setMicEnabled(true);
    btn.innerText = "Init";
    muteBtn.style.display = "none";
    transcribeBtn.style.display = "none";
    status.innerText = "Transcribe all...";
    transcribeBtn.disabled = true;
    btn.disabled = true;
    await google.colab.kernel.invokeFunction("notebook.transcribe_all", [], {});
    status.innerText = "Done";
    btn.disabled = false;
    transcribeBtn.disabled = false;
};

function isSpeech(dataArray) {
    const binHz = audioCtx.sampleRate / (analyser.frequencyBinCount * 2);
    const voiceLow  = Math.floor(80   / binHz);
    const voiceHigh = Math.floor(4000 / binHz);
    let total = 0, voice = 0;
    for (let i = 0; i < dataArray.length; i++) {
        total += dataArray[i];
        if (i >= voiceLow && i <= voiceHigh) voice += dataArray[i];
    }
    return (total / dataArray.length) > VOLUME_THRESHOLD &&
           (total > 0 ? voice / total : 0) > SPEECH_RATIO;
}

function saveChunk() {
    if (chunks.length === 0) return;
    const blob = new Blob(chunks, {type: "audio/webm"});
    chunks = [];
    const reader = new FileReader();
    reader.readAsDataURL(blob);
    reader.onloadend = async () => {
        await google.colab.kernel.invokeFunction("notebook.save_chunk", [reader.result], {});
    };
}

function stopSpeech(save = true) {
    speaking = false;
    clearTimeout(silenceTimer);
    silenceTimer = null;
    if (mediaRecorder && mediaRecorder.state === "recording") {
        if (!save) chunks = [];
        mediaRecorder.stop();
    }
}

function startSpeech() {
    speaking = true;
    chunks = [];
    mediaRecorder = new MediaRecorder(liveStream);
    mediaRecorder.ondataavailable = e => { if (e.data.size > 0) chunks.push(e.data); };
    mediaRecorder.onstop = () => { if (chunks.length > 0) saveChunk(); };
    mediaRecorder.start(100);
    status.innerText = "Speaking...";
}

function vadLoop() {
    if (!analyser || !liveRecording) return;
    const data = new Uint8Array(analyser.frequencyBinCount);
    analyser.getByteFrequencyData(data);
    if (!muted && isSpeech(data)) {
        if (!speaking) startSpeech();
        clearTimeout(silenceTimer);
        silenceTimer = setTimeout(() => {
            if (speaking) {
                stopSpeech(true);
                if (!muted) status.innerText = "Listening...";
            }
        }, SILENCE_DELAY_MS);
    }
    animFrame = requestAnimationFrame(vadLoop);
}

btn.onclick = async () => {
    if (liveRecording) {
        liveRecording = false;
        muted = false;
        cancelAnimationFrame(animFrame);
        if (speaking) stopSpeech(false);
        setMicEnabled(true);
        btn.innerText = "INIT";
        muteBtn.style.display = "none";
        muteBtn.innerText = "MUTE";
        status.innerText = "";
        return;
    }

    if (!liveStream) {
        try {
            liveStream = await navigator.mediaDevices.getUserMedia({ audio: true });
        } catch(e) {
            alert("Denied MIC: " + e.message);
            return;
        }
    }

    audioCtx = new AudioContext();
    const source = audioCtx.createMediaStreamSource(liveStream);
    analyser = audioCtx.createAnalyser();
    analyser.fftSize = 1024;
    source.connect(analyser);

    liveRecording = true;
    muted = false;
    btn.innerText = "Stop";
    muteBtn.style.display = "inline-block";
    transcribeBtn.style.display = "inline-block";
    status.innerText = "Listening...";
    vadLoop();
};
'''))

Loading whisper large...
Ready



<IPython.core.display.Javascript object>

🎙 Grabación 1 guardada
🎙 Grabación 2 guardada
🎙 Grabación 3 guardada
🎙 Grabación 4 guardada
🎙 Grabación 5 guardada

🔄 Transcribiendo 5 grabaciones...

  [1] Cabe destacar que en zonas de tormenta se podrían registrar intensa actividad eléctrica.
  [2] Acompañadas de lluvias abundantes en cortos periodos de tiempo.
  [3] Ocasionalmente caída de granizo y rayos de vientos fuertes.
  [4] Se esperan mejoras temporales durante la validez de la advertencia.
  [5] Se continuará monitoreando la situación y se informará ante eventos o cambios.

📝 Cabe destacar que en zonas de tormenta se podrían registrar intensa actividad eléctrica.. Acompañadas de lluvias abundantes en cortos periodos de tiempo.. Ocasionalmente caída de granizo y rayos de vientos fuertes.. Se esperan mejoras temporales durante la validez de la advertencia.. Se continuará monitoreando la situación y se informará ante eventos o cambios.

🎙 Grabación 1 guardada
🎙 Grabación 2 guardada

🔄 Transcribiendo 2 grabaciones...

  [1] Der